In [1]:
import re
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity
import hdbscan
import cuml
import joblib

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [2]:
TOOL_REPLACEMENTS = {
    r'\b(kafka|rabbitmq|nats|pulsar|sqs|sns|activemq|zeromq)\b': 'msg_queue',
    r'\b(postgres|postgresql|mysql|mongodb|redis|cassandra|dynamodb|elasticsearch|clickhouse|snowflake|redshift|bigquery)\b': 'database',
    r'\b(kubernetes|k8s|eks|gke|aks|nomad|ecs)\b': 'container_platform',
    r'\b(terraform|pulumi|cloudformation|ansible|helm|argocd|flux)\b': 'iac_tool',
    r'\b(jenkins|github.actions|gitlab.ci|circleci|tekton|drone)\b': 'cicd_tool',
    r'\b(aws|gcp|azure|s3|ec2|rds|gcs|blob)\b': 'cloud_provider',
    r'\b(fastapi|django|flask|spring|rails|express|nestjs|react|flutter|kotlin|rust|golang|python|java|typescript)\b': 'app_framework',
    r'\b(mlflow|kubeflow|airflow|spark|flink|tensorflow|pytorch|xgboost)\b': 'ml_tool',
    r'\b(datadog|prometheus|grafana|pagerduty|newrelic|splunk|kibana|jaeger)\b': 'observability_tool',
}

In [3]:
INTENT_KEYWORDS = {
    'failing', 'crashed', 'error', 'timeout', 'unreachable', 'rejected',
    'blocked', 'stuck', 'down', 'unavailable', 'corrupted', 'missing',
    'expired', 'invalid', 'denied', 'exhausted', 'leak', 'duplicate',
    'slow', 'latency', 'lag', 'spike', 'overflow', 'oom', 'killed',
    'rotate', 'reset', 'restore', 'rollback', 'restart', 'grant', 'revoke',
    'update', 'migrate', 'deploy', 'provision', 'debug', 'investigate',
    'production', 'outage', 'blocking', 'degraded', 'data.loss', 'breach',
}

In [4]:
def normalize_tools(text: str) -> str:
    text = text.lower()
    for pattern, replacement in TOOL_REPLACEMENTS.items():
        text = re.sub(pattern, replacement, text)
    return text

In [5]:
def extract_intent_text(text: str) -> str:
    """Remove version numbers, commit hashes, IPs, ports, incident/PR numbers."""
    text = re.sub(r'\bv\d+[\.\d]+\b', '', text)
    text = re.sub(r'\b[a-f0-9]{7,40}\b', '', text)
    text = re.sub(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', '', text)
    text = re.sub(r':\d{2,5}\b', '', text)
    text = re.sub(r'\bpr\s*#?\d+\b', '', text)
    text = re.sub(r'\binc-?\d+\b', '', text)
    return text.strip()


In [10]:
df = pd.read_csv("/kaggle/input/datasets/crazyapil/train-dataset-3/train_data.csv")

In [11]:
df = df.drop_duplicates(subset=['description'], keep='first')

print(f"Original rows: {len(df)}")
print(f"After dedup on cleaned_text: {len(df)}")

Original rows: 10147
After dedup on cleaned_text: 10147


In [12]:
descriptions = df['description'].tolist()

# Tool abstraction + intent extraction (for vocabulary building)
processed_texts = [extract_intent_text(normalize_tools(d)) for d in descriptions]

In [13]:
temp_vec = TfidfVectorizer(
    stop_words='english',
    max_features=500,
    ngram_range=(1, 2),
    sublinear_tf=True
)
temp_vec.fit(processed_texts)
all_features = set(temp_vec.get_feature_names_out())

In [14]:
tool_tokens = set(TOOL_REPLACEMENTS.values())
noise_words = {'pr', 'inc', 'v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8', 'v9', 'v10',
               'v11', 'v12', 'v13', 'v14', 'v15', 'v16', 'v17', 'v18', 'v19', 'v20',
               'commit', 'hash', 'ip', 'port', 'pr_number', 'inc_number'}
final_vocab = sorted(all_features - tool_tokens - noise_words)
print(f"Final vocabulary size: {len(final_vocab)}")

Final vocabulary size: 488


In [15]:
intent_vectorizer = TfidfVectorizer(
    vocabulary=final_vocab,
    sublinear_tf=True,
    norm='l2'
)
intent_tfidf = intent_vectorizer.fit_transform(processed_texts)

In [16]:
INTENT_PREFIX = (
    "Represent the core problem type and required action of this support ticket, "
    "ignoring specific tool names: "
)

In [17]:
%%capture
!pip install sentence-transformers

In [18]:
from sentence_transformers import SentenceTransformer
import torch

In [19]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [20]:
model_name = "BAAI/bge-m3"

In [21]:
embedder = SentenceTransformer(model_name)
embedder.to(device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 8192, 'do_lower_case': False, 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [22]:
emb_raw = embedder.encode(descriptions, batch_size=64, normalize_embeddings=True)

emb_intent = embedder.encode(
    [INTENT_PREFIX + d for d in descriptions],
    batch_size=64,
    normalize_embeddings=True
)

In [23]:
umap_tmp = cuml.UMAP(n_components=15, n_neighbors=35, min_dist=0.1,
                     metric='cosine', random_state=42)


In [24]:
best_validity = -1
best_ratio = 0.6
blend_ratios = [0.3, 0.4, 0.5, 0.6, 0.7]
for w in blend_ratios:
    blended = emb_raw * (1-w) + emb_intent * w
    blended /= np.linalg.norm(blended, axis=1, keepdims=True)
    hyb = np.hstack([blended * 0.75,
                     intent_tfidf.toarray() * 0.25])
    reduced_tmp = umap_tmp.fit_transform(hyb)
    clusterer_tmp = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=3,
                                    metric='euclidean', cluster_selection_method='eom',
                                    cluster_selection_epsilon=0.1, gen_min_span_tree=True)
    labels_tmp = clusterer_tmp.fit_predict(reduced_tmp)
    validity = clusterer_tmp.relative_validity_
    print(f"  Ratio {w:.1f}: DBCV = {validity:.4f}")
    if validity > best_validity:
        best_validity = validity
        best_ratio = w

print(f"Best blend ratio: {best_ratio:.1f} (validity: {best_validity:.4f})")

  Ratio 0.3: DBCV = 0.3102
  Ratio 0.4: DBCV = 0.2680
  Ratio 0.5: DBCV = 0.2368
  Ratio 0.6: DBCV = 0.2708
  Ratio 0.7: DBCV = 0.2690
Best blend ratio: 0.3 (validity: 0.3102)


In [25]:
emb_blended = emb_raw * (1-best_ratio) + emb_intent * best_ratio
emb_blended /= np.linalg.norm(emb_blended, axis=1, keepdims=True)

hybrid_emb = np.hstack([
    emb_blended * 0.75,
    intent_tfidf.toarray() * 0.25
])

hybrid_emb /= np.linalg.norm(hybrid_emb, axis=1, keepdims=True)

In [26]:
import cupy as cp

In [27]:
umap_model = cuml.UMAP(
    n_components=15,
    n_neighbors=35,  
    min_dist=0.1,        
    metric='cosine',
    random_state=42
)
reduced = umap_model.fit_transform(hybrid_emb)
reduced_np = cp.asnumpy(reduced)

In [28]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=18,    
    min_samples=3,
    metric='euclidean',
    cluster_selection_method='eom',
    cluster_selection_epsilon=0.1,
    prediction_data=True
)
labels_raw = clusterer.fit_predict(reduced_np)

In [29]:
n_clusters = len(set(labels_raw)) - (1 if -1 in labels_raw else 0)
noise_pct = (labels_raw == -1).sum() / len(labels_raw) * 100
print(f"Raw HDBSCAN: {n_clusters} clusters, {noise_pct:.1f}% noise")

Raw HDBSCAN: 119 clusters, 24.2% noise


In [30]:
def get_medoid(embs, indices):
    """indices are global positions; returns medoid index (global)."""
    cluster_embs = embs[indices]
    dist_mat = cosine_distances(cluster_embs)
    medoid_in = np.argmin(dist_mat.mean(axis=1))
    return indices[medoid_in]

In [31]:
embeddings_np = emb_raw 

In [32]:
unique_labels = set(labels_raw) - {-1}
medoid_map = {}
for lab in unique_labels:
    idxs = np.where(labels_raw == lab)[0]
    medoid_map[lab] = embeddings_np[get_medoid(embeddings_np, idxs)] 

In [33]:
labels_reassigned = labels_raw.copy()
noise_indices = np.where(labels_reassigned == -1)[0]
THRESHOLD = 0.35  # cosine distance, lower = stricter

for idx in noise_indices:
    vec = embeddings_np[idx]
    best_lab, best_dist = -1, 1.0
    for lab, med in medoid_map.items():
        d = cosine_distances([vec], [med])[0,0]
        if d < best_dist:
            best_dist, best_lab = d, lab
    if best_lab != -1 and best_dist < THRESHOLD:
        labels_reassigned[idx] = best_lab

n_noise_after = (labels_reassigned == -1).sum()
print(f"After reassignment: {n_noise_after} noise points ({n_noise_after/len(labels_reassigned)*100:.1f}%)")

After reassignment: 1915 noise points (18.9%)


In [34]:
MIN_KEEP = 10
labels_final = labels_reassigned.copy()
unique_final = set(labels_final) - {-1}
sizes = {lab: (labels_final == lab).sum() for lab in unique_final}

In [35]:
for small_lab in sorted([l for l, sz in sizes.items() if sz < MIN_KEEP]):
    idx_small = np.where(labels_final == small_lab)[0]
    med_small = embeddings_np[get_medoid(embeddings_np, idx_small)]
    best_target, best_dist = -1, 1.0
    for large_lab in [l for l, sz in sizes.items() if sz >= MIN_KEEP]:
        idx_large = np.where(labels_final == large_lab)[0]
        med_large = embeddings_np[get_medoid(embeddings_np, idx_large)]
        d = cosine_distances([med_small], [med_large])[0,0]
        if d < best_dist:
            best_dist, best_target = d, large_lab
    if best_target != -1 and best_dist < 0.5:
        labels_final[labels_final == small_lab] = best_target
        print(f"  Merged tiny cluster {small_lab} ({sizes[small_lab]} tix) → {best_target}")
    else:
        labels_final[labels_final == small_lab] = -1
        print(f"  Marked tiny cluster {small_lab} ({sizes[small_lab]} tix) as noise (no close neighbour)")

In [36]:
unique_final = set(labels_final) - {-1}
sizes_final = {lab: (labels_final == lab).sum() for lab in unique_final}
print(f"Final clusters: {len(unique_final)}, final noise: {(labels_final == -1).sum()}")


Final clusters: 119, final noise: 1915


In [37]:
from sklearn.metrics.pairwise import cosine_distances
from sklearn.cluster import AgglomerativeClustering

In [38]:
fine_labels = labels_final   
unique_fine = sorted([l for l in set(fine_labels) if l != -1])

medoid_embs = []          
medoid_labels = []        

In [39]:
def get_medoid(embs, indices):
    cluster_embs = embs[indices]
    dist_mat = cosine_distances(cluster_embs)
    medoid_in = np.argmin(dist_mat.mean(axis=1))
    return indices[medoid_in]

In [40]:
for cid in unique_fine:
    idxs = np.where(fine_labels == cid)[0]
    medoid_idx = get_medoid(embeddings_np, idxs)
    medoid_embs.append(embeddings_np[medoid_idx])
    medoid_labels.append(cid)

medoid_embs = np.array(medoid_embs)

In [41]:
N_META = 60

agg = AgglomerativeClustering(
    n_clusters=N_META,
    metric='cosine',
    linkage='average'
)
meta_labels = agg.fit_predict(medoid_embs)

In [42]:
fine_to_meta = {fine: meta for fine, meta in zip(medoid_labels, meta_labels)}

meta_final = np.full_like(fine_labels, -1, dtype=int)
for idx in range(len(fine_labels)):
    if fine_labels[idx] != -1:
        meta_final[idx] = fine_to_meta[fine_labels[idx]]

In [43]:
for meta_id in sorted(set(meta_final) - {-1}):
    idxs = np.where(meta_final == meta_id)[0]
    if len(idxs) < 5:
        continue
    medoid_idx = get_medoid(embeddings_np, idxs)
    medoid_vec = embeddings_np[medoid_idx].reshape(1, -1)
    cluster_embs = embeddings_np[idxs]
    dists = cosine_distances(cluster_embs, medoid_vec).ravel()
    sorted_order = np.argsort(dists)
    top_k = min(10, len(idxs))
    top_indices = idxs[sorted_order[:top_k]]

    print(f"\n{'='*60}")
    print(f"Meta‑Cluster {meta_id} – {len(idxs)} tickets")
    print(f"{'='*60}")
    for rank, idx in enumerate(top_indices, 1):
        preview = descriptions[idx].replace('\n', ' ').strip()[:200]
        print(f" {rank:2d}. {preview}")


Meta‑Cluster 0 – 138 tickets
  1. Spring Boot @ConfigurationProperties validation isn't failing on invalid values because @Validated is missing on the class. Invalid config loads silently. We need to add annotation, implement constrai
  2. Spring Boot @ConfigurationProperties validation isn't failing on invalid values because @Validated is missing. Invalid config loads silently. We need to add annotation, implement constraint validators
  3. @ConfigurationProperties validation isn't failing on invalid values because @Validated is missing on the class. Invalid config loads silently. We need to add annotation, implement constraint validator
  4. @ConfigurationProperties isn't validating because @Validated is missing and constraints are ignored. We need to add annotation, implement validators, and add config validation tests.
  5. @Validated isn't working because constraint validator isn't registered in Spring context. We need to add @Component to validator, validate registration, and ad

In [44]:
meta_label_map = {
    0:  "Spring Boot Configuration Property Validation & Binding Failures",
    1:  "External Webhook Delivery Timeouts & Retry Backlogs from Partner APIs",
    2:  "React/Flutter UI Rendering Issues – Suspense Fallbacks, Scroll Physics & Layout Jumps",
    3:  "Local Development Port Conflicts & Orphaned Container Cleanup (Docker Compose)",
    4:  "Angular Signals API Misuse – Effect Loops, Computed Recalculation & SSR Safety",
    5:  "Celery/Django Task Configuration Anti‑Patterns – Request Context, Logging & Backend Errors",
    6:  "Missing Test Coverage for API Gateway Idempotency, Request Validation & Retry Behaviour",
    7:  "Auto‑Scaling Policy Failures – HPA Not Triggering Due to Metrics/Certificate Issues",
    8:  "LLM Prompt Injection Vulnerabilities & Template Versioning Gaps",
    9:  "Infrastructure State Lock Contention & Stale Lock Force‑Unlock Requests (Terraform)",
    10: "Message Consumer Lag & Processing Backlogs from Slow Downstream Services (Kafka/Streaming)",
    11: "Application Session Timeouts, Concurrency Control & Shared Resource Contention",
    12: "Job Scheduler Unresponsiveness – DAG Trigger Failures & Metadata DB Timeouts (Airflow)",
    13: "GPU Training Resource Exhaustion – CUDA Out of Memory, Memory Fragmentation & NCCL Timeouts",
    14: "Jenkins Pipeline Authentication & Connection Failures – ECR, npm, Terraform, Helm",
    15: "Spark Job Failures – Executor OOM, Shuffle Disk Full & Heartbeat Timeouts",
    16: "Ansible Dynamic Inventory Failures – IAM Permission Changes Blocking EC2 Discovery",
    17: "Local Docker Disk Space Bloat – Unpruned Volumes, Networks & Build Cache Accumulation",
    18: "Grafana Dashboard Metric Discrepancies & Data Gaps After Prometheus/Thanos Migration",
    19: "External API Rate Limiting (429) – Credit Bureau, Tax, News & Market Data Providers",
    20: "Cache Memory Pressure, Fragmentation & Aggressive Eviction Causing API Latency Spikes (Redis)",
    21: "Search Index Performance & Indexing Throughput Degradation During Peak Audit Ingestion (Elasticsearch)",
    22: "Internal DNS Service Resolution Failures – CoreDNS NXDOMAIN/SERVFAIL & CNI Policy Blocks",
    23: "Accessibility Audit Failures – Modal Focus Traps, Keyboard Navigation & Screen Reader Gaps",
    24: "Internal TLS/SSL Certificate Expiry & Cert‑Manager Auto‑Renewal Failures",
    25: "Primary‑Replica Data Replication Lag & Staleness Causing Risk/Fraud Calculation Errors (Postgres)",
    26: "Flutter/Native Memory Leaks – EventChannel, ViewModel & Coroutine Lifecycle Issues",
    27: "Datadog Agent Authentication Failures – API Key Rotation, 401/403 Errors & Missing Metrics",
    28: "iOS/macOS CI Code Signing & Provisioning Profile Expiry Blocking Mobile Builds",
    29: "Blue‑Green Deployment Traffic Switch Failures – Health Check, Probe & Ingress Misconfig",
    30: "Helm Chart Installation & Upgrade Failures – Stale Repos, Deprecated Charts & CRD Conflicts",
    31: "Frontend Build Memory Exhaustion – JavaScript Heap OOM in CI Pipelines",
    32: "Local IDE & Python Virtual Environment Setup Issues (VS Code, debugpy, dotenv)",
    33: "Kotlin Multiplatform / Gradle Build Cache Misses, KSP Performance & Compiler Issues",
    34: "ML Feature Engineering Pipeline Errors – NaN Values, Feature Mismatch & Data Alignment",
    35: "Security Audit Logging & Monitoring Gaps – auditd, Vault, CloudTrail, Splunk Misconfigurations",
    36: "RAG Pipeline Retrieval Quality Issues – Chunking, Deduplication & Multi‑Hop Failures",
    37: "Accidentally Deleted Kubernetes ConfigMaps / Feature Flags – Urgent Restore Requests",
    38: "Monitoring & Administration UI Outages After Backend Migration (Airflow Flower, MLflow)",
    39: "Scheduled Job Timezone & Daylight Saving Time Misconfiguration (CronJob, Cron Expressions)",
    40: "Django Session & Authentication Configuration Errors – Engine, Cookie, Backend Misconfigurations",
    41: "Terraform Apply IAM Permission & API Rate Limit Denials (403 Forbidden, STS AssumeRole)",
    42: "BGP Session Flapping & Cross‑Region Packet Loss Affecting Trading Connectivity",
    43: "Critical Security Vulnerability Remediation – Command Injection, Log4Shell & Dependency CVEs",
    44: "Docker Desktop High CPU Usage & Battery Drain on macOS (HyperKit Resource Hogging)",
    45: "Kubernetes Network Policy Blocking Legitimate Egress – ML Inference, Compliance & Redis",
    46: "Docker Build COPY Instruction Failures – Missing Files, Wrong Build Context & Symlinks",
    47: "ETL Data Type Mismatch, Null Value & Duplicate Key Errors During Warehouse Loads",
    48: "Laptop Hardware Malfunctions & Peripheral Failures (Display, USB, Trackpad, Monitor)",
    49: "ML Model Serving OOM Kills – TensorFlow, Fraud/ Risk Model Pods Exceeding Memory Limits",
    50: "Flutter/React Native App Crashes on Android 14 / iOS 17 Due to Permissions & SDK Updates",
    51: "Source Code Collaboration Issues – Merge Conflicts, Rebase Failures & Push Rejections (Git)",
    52: "SSH Key Management, Bastion Access & Agent Forwarding Configuration",
    53: "Temporary Production Access Requests for Kafka Topics, Logs & Audit Trails",
    54: "Password Reset & MFA Authentication Issues – Expired Links, Missing Emails & Token Rejections",
    55: "Log Aggregation Pipeline Issues – Filebeat Drops, Logstash Heap Exhaustion & Indexing Gaps (ELK)",
    56: "Missing Payment/Bank/CRM Webhook Callbacks – TLS, URL Mismatch & Signature Verification",
    57: "FastAPI Middleware Latency Overhead from Request Tracing, Body Serialization & Logging",
    58: "Django REST Framework Serializer Validation Errors – Field Visibility, Validators & Bulk Operations",
    59: "kubectl Operational Pain – Evicted Pods, Slow Commands, Lost Logs & Exec/Port‑Forward Failures",
}

In [45]:
cluster_to_department = {
    0:  "Application Development",
    1:  "External Integrations & API Management",
    2:  "Application Development",
    3:  "Developer Experience & Tooling",
    4:  "Application Development",
    5:  "Application Development",
    6:  "Application Development",
    7:  "DevOps / Platform Engineering",
    8:  "AI / ML",
    9:  "Deployment & IaC",
    10: "Data Streaming & Pipeline",
    11: "Application Development",
    12: "Deployment & IaC",
    13: "AI / ML",
    14: "CI/CD & Build Systems",
    15: "Data Streaming & Pipeline",
    16: "Deployment & IaC",
    17: "Developer Experience & Tooling",
    18: "Monitoring & Observability",
    19: "External Integrations & API Management",
    20: "Database & Data Engineering",
    21: "Monitoring & Observability",
    22: "Networking & Connectivity",
    23: "Application Development",
    24: "Security",
    25: "Database & Data Engineering",
    26: "Application Development",
    27: "Monitoring & Observability",
    28: "CI/CD & Build Systems",
    29: "Deployment & IaC",
    30: "Deployment & IaC",
    31: "CI/CD & Build Systems",
    32: "Developer Experience & Tooling",
    33: "Application Development",
    34: "AI / ML",
    35: "Security",
    36: "AI / ML",
    37: "Deployment & IaC",
    38: "Monitoring & Observability",
    39: "Deployment & IaC",
    40: "Application Development",
    41: "Deployment & IaC",
    42: "Networking & Connectivity",
    43: "Security",
    44: "Developer Experience & Tooling",
    45: "Networking & Connectivity",
    46: "CI/CD & Build Systems",
    47: "Data Streaming & Pipeline",
    48: "Hardware / IT Support",
    49: "AI / ML",
    50: "Application Development",
    51: "Developer Experience & Tooling",
    52: "Security",
    53: "Access & Permissions",
    54: "Access & Permissions",
    55: "Monitoring & Observability",
    56: "External Integrations & API Management",
    57: "Monitoring & Observability",
    58: "Application Development",
    59: "DevOps / Platform Engineering",
}

In [46]:
df['cluster_id'] = labels_final

df['meta_cluster_id'] = meta_final

df['meta_label_name'] = df['meta_cluster_id'].map(meta_label_map)

df['department'] = df['meta_cluster_id'].map(cluster_to_department)

In [47]:
noise_mask = df['meta_cluster_id'] == -1
df.loc[noise_mask, 'meta_label_name'] = 'Uncategorised / Rare Issues'
df.loc[noise_mask, 'department'] = 'Uncategorised'

In [48]:
df.head()

,description,priority,parity,cluster_id,meta_cluster_id,meta_label_name,department
0,I triggered a Jenkins deployment for the settl...,high,NaN,74,29,Blue‑Green Deployment Traffic Switch Failures ...,Deployment & IaC
1,Our Airflow ETL pipeline for compliance report...,high,NaN,-1,-1,Uncategorised / Rare Issues,Uncategorised
2,I need RBAC access to the fraud-detection name...,medium,NaN,73,41,Terraform Apply IAM Permission & API Rate Limi...,Deployment & IaC
3,The GitHub Actions workflow for the payment-ro...,medium,NaN,89,28,iOS/macOS CI Code Signing & Provisioning Profi...,CI/CD & Build Systems
4,Grafana dashboards for the trading-engine are ...,medium,NaN,28,18,Grafana Dashboard Metric Discrepancies & Data ...,Monitoring & Observability


In [49]:
import joblib

In [50]:
valid_meta = [m for m in set(meta_final) if m != -1]

medoid_vectors = []
medoid_ids = []

for meta_id in sorted(valid_meta):
    idxs = np.where(meta_final == meta_id)[0]
    medoid_idx = get_medoid(embeddings_np, idxs)  
    medoid_vectors.append(embeddings_np[medoid_idx])
    medoid_ids.append(meta_id)

In [51]:
medoid_vectors = np.array(medoid_vectors)

In [52]:
# joblib.dump(medoid_vectors, 'meta_medoids.pkl')
# joblib.dump(medoid_ids, 'meta_ids.pkl')
# joblib.dump(meta_label_map, 'meta_label_map.pkl')

In [53]:
from sklearn.metrics.pairwise import cosine_similarity


In [54]:
medoid_vectors = medoid_vectors   
medoid_ids = medoid_ids           
meta_label_map = meta_label_map      

Priority

In [55]:
df

,description,priority,parity,cluster_id,meta_cluster_id,meta_label_name,department
0,I triggered a Jenkins deployment for the settl...,high,NaN,74,29,Blue‑Green Deployment Traffic Switch Failures ...,Deployment & IaC
1,Our Airflow ETL pipeline for compliance report...,high,NaN,-1,-1,Uncategorised / Rare Issues,Uncategorised
2,I need RBAC access to the fraud-detection name...,medium,NaN,73,41,Terraform Apply IAM Permission & API Rate Limi...,Deployment & IaC
3,The GitHub Actions workflow for the payment-ro...,medium,NaN,89,28,iOS/macOS CI Code Signing & Provisioning Profi...,CI/CD & Build Systems
4,Grafana dashboards for the trading-engine are ...,medium,NaN,28,18,Grafana Dashboard Metric Discrepancies & Data ...,Monitoring & Observability
...,...,...,...,...,...,...,...
13171,I noticed LVM logical volume extended but file...,high,NaN,43,25,Primary‑Replica Data Replication Lag & Stalene...,Database & Data Engineering
13172,I noticed Nginx proxy buffering causing 504 Ga...,medium,NaN,108,1,External Webhook Delivery Timeouts & Retry Bac...,External Integrations & API Management
13173,I discovered journald logs consuming 78% of /v...,medium,NaN,43,25,Primary‑Replica Data Replication Lag & Stalene...,Database & Data Engineering
13174,I noticed SSH key authentication failing after...,medium,NaN,75,52,"SSH Key Management, Bastion Access & Agent For...",Security


In [56]:
df['priority'].value_counts()

priority
medium    4171
high      3976
low       1999
Name: count, dtype: int64

In [57]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

In [58]:
mask = df['priority'].notna()
X = embeddings_np[mask]        
y = df.loc[mask, 'priority'].values

In [59]:
le = LabelEncoder()
y_enc = le.fit_transform(y)

In [60]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

In [61]:
model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric='mlogloss'
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

        high       0.77      0.78      0.77       795
         low       0.88      0.60      0.71       400
      medium       0.69      0.78      0.73       835

    accuracy                           0.74      2030
   macro avg       0.78      0.72      0.74      2030
weighted avg       0.76      0.74      0.74      2030



In [62]:
import re

URGENCY_PATTERNS = {
    'production_down':    r'\b(production|prod)\b.{0,30}\b(down|outage|failing|crashed)\b',
    'data_loss':          r'\b(data.loss|corruption|duplicate.charge|double.charge)\b',
    'revenue_impact':     r'\b(revenue|customer.facing|blocking.all|cannot.trade|cannot.pay)\b',
    'security_critical':  r'\b(breach|exploit|injection|exposure|compromised)\b',
    'rollback_needed':    r'\b(roll.?back|revert|immediate)\b',
    'local_only':         r'\b(local|localhost|my.laptop|my.machine|dev.env)\b',
    'cosmetic':           r'\b(typo|tooltip|dark.mode|font|icon|label)\b',
    'access_request':     r'\b(need.access|request.access|read.only|temporary.access)\b',
    'how_to':             r'\b(how.do.i|how.can.i|is.there.a.way|can.we.add)\b',
    'quota_request':      r'\b(quota|limit.increase|more.resources)\b',
}

In [63]:
def extract_urgency_features(description: str) -> np.ndarray:
    text = description.lower()
    features = []
    for pattern in URGENCY_PATTERNS.values():
        features.append(1.0 if re.search(pattern, text) else 0.0)
    features.append(len(description) / 1000.0)             
    features.append(1.0 if 'error' in text else 0.0)
    features.append(1.0 if 'exception' in text else 0.0)
    features.append(text.count('%') / 10.0)                 
    return np.array(features, dtype=np.float32)

urgency_features = np.array([
    extract_urgency_features(d) for d in descriptions
])
print(f"Urgency feature shape: {urgency_features.shape}")


Urgency feature shape: (10147, 14)


In [64]:
X_combined = np.hstack([embeddings_np, urgency_features])

In [65]:
mask = df['priority'].notna()
X = X_combined[mask]
y = df.loc[mask, 'priority'].values
le = LabelEncoder()
y_enc = le.fit_transform(y)

In [66]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

In [67]:
from collections import Counter
class_counts = Counter(y_train)
total = sum(class_counts.values())
scale_pos_weight = {
    cls: total / (len(class_counts) * count)
    for cls, count in class_counts.items()
}
sample_weights = np.array([scale_pos_weight[y] for y in y_train])

In [68]:
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss'
)
model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    verbose=False
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [69]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

        high       0.77      0.77      0.77       795
         low       0.79      0.69      0.74       400
      medium       0.69      0.73      0.71       835

    accuracy                           0.74      2030
   macro avg       0.75      0.73      0.74      2030
weighted avg       0.74      0.74      0.74      2030



In [70]:
def classify_ticket(description, threshold=0.58, use_llm_fallback=True):
    raw_emb = embedder.encode([description], normalize_embeddings=True).astype('float32')
    urg_feat = extract_urgency_features(description).reshape(1, -1)
    combined_emb = np.hstack([raw_emb, urg_feat])        

    pred_enc = model.predict(combined_emb)[0]
    priority = le.inverse_transform([pred_enc])[0]

    sims = cosine_similarity(raw_emb, medoid_vectors).flatten()
    best_idx = np.argmax(sims)
    best_score = sims[best_idx]

    if best_score >= threshold:
        meta_id = medoid_ids[best_idx]
        label = meta_label_map.get(meta_id, 'Unknown')
        return {
            'label': label,
            'confidence': round(best_score, 4),
            'priority': priority,
            'source': 'medoid',
            'solution': None
        }

    return {
        'label': 'Uncategorized / Rare Issues',
        'confidence': round(best_score, 4),
        'priority': priority,
        'source': 'none',
        'solution': None
    }

In [71]:
ticket = "After a Helm upgrade, the ConfigMap for feature flags didn''t reload and the pods are still using old values."
result = classify_ticket(ticket)
print(result['label'])     
print(result['priority'])   
print(result['confidence'])

Accidentally Deleted Kubernetes ConfigMaps / Feature Flags – Urgent Restore Requests
medium
0.6717


In [72]:
mask_valid = (meta_final != -1)                
inference_embs = embeddings_np[mask_valid]    
inference_labels = meta_final[mask_valid]     
inference_descs = df['description'].values[mask_valid]

In [73]:
def classify_ticket_knn(description, k=5, sim_threshold=0.60, use_llm_fallback=True):
    raw_emb = embedder.encode([description], normalize_embeddings=True).astype('float32')
    urg_feat = extract_urgency_features(description).reshape(1, -1)
    combined_emb = np.hstack([raw_emb, urg_feat])  

    pred_enc = model.predict(combined_emb)[0]
    priority = le.inverse_transform([pred_enc])[0]

    sims = cosine_similarity(raw_emb, inference_embs).flatten()
    top_k_idx = np.argsort(sims)[-k:][::-1]
    top_k_sims = sims[top_k_idx]
    top_k_labels = inference_labels[top_k_idx]

    best_sim = top_k_sims[0]

    label_counts = Counter(top_k_labels)
    most_common = label_counts.most_common()
    majority_id, majority_votes = most_common[0]

 
    if len(most_common) > 1 and most_common[1][1] == majority_votes:
        tied_ids = [lbl for lbl, cnt in most_common if cnt == majority_votes]
        avg_sim = {}
        for lbl in tied_ids:
            mask = top_k_labels == lbl
            avg_sim[lbl] = np.mean(top_k_sims[mask])
        majority_id = max(avg_sim, key=avg_sim.get)

    label = meta_label_map.get(majority_id, 'Unknown')
    confidence = round(best_sim, 4)  


    solution = None
    source = 'knn'
    if best_sim < sim_threshold and use_llm_fallback:
        try:
            llm_result = llm_classify_and_solve(description)
            llm_label = llm_result.get('label', 'Uncategorized / Rare Issues')
            if llm_label not in meta_label_map.values():
                llm_label = 'Uncategorized / Rare Issues'
            label = llm_label
            solution = llm_result.get('solution')
            source = 'llm'
        except Exception:
            source = 'knn_low_conf'

    similar = []
    for i, idx in enumerate(top_k_idx):
        similar.append({
            'similarity': round(float(top_k_sims[i]), 4),
            'description': inference_descs[idx][:200] if 'inference_descs' in dir() else ''
        })

    return {
        'label': label,
        'confidence': confidence,
        'priority': priority,
        'source': source,
        'solution': solution,
        'similar_tickets': similar
    }


In [74]:
ticket = "The new office Wi-Fi keeps dropping every hour, making it impossible to join video calls"
result = classify_ticket_knn(ticket)
print(result['label'])     
print(result['priority'])   
print(result['confidence'])

Laptop Hardware Malfunctions & Peripheral Failures (Display, USB, Trackpad, Monitor)
medium
0.6527


In [83]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 2.6 MB/s eta 0:00:00a 0:00:01


In [84]:
import os
from groq import Groq

In [85]:
groq_client = Groq(api_key="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")
LLM_MODEL = 'qwen/qwen3-32b'

In [86]:
def llm_generate_solution(description: str, label: str, priority: str,
                          similar_tickets: list) -> str:
    """Generate a solution for a classified ticket (medoid path)."""
    context = "\n".join([
        f"- [{t['similarity']:.2f}] {t['description']}"
        for t in similar_tickets[:3]
    ])
    prompt = f"""You are a senior IT operations expert. A support ticket has been classified as follows:

TICKET: {description}
CATEGORY: {label}
PRIORITY: {priority}

Most similar past tickets:
{context}

Write a clear, step‑by‑step resolution plan (under 150 words). Return ONLY a JSON object with the key "solution". /no_think"""
    try:
        response = groq_client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model=LLM_MODEL,
            temperature=0.2,
            max_tokens=400,
        )
        raw = response.choices[0].message.content
        json_str = extract_json(raw) 
        result = json.loads(json_str)
        return result.get("solution") or "A solution could not be generated."
    except Exception:
        return "Solution generation temporarily unavailable."

In [87]:
def extract_json(text: str) -> str:
    """
    Extract a JSON object from raw LLM output that may contain markdown fences,
    thinking blocks, or trailing text.
    """
    # Remove thinking blocks if present
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    # Try to find a JSON object
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        return m.group(0)
    return text.strip()

In [88]:
def llm_fallback_free(description: str, max_retries: int = 2) -> dict:
    prompt = f"""You are an expert IT support triage assistant. For the ticket below, provide a short, descriptive category label (e.g., "Office Wi‑Fi Connectivity Issue", "Database Credential Expiry", "Build Memory Exhaustion") and a step‑by‑step solution (under 150 words).

Return ONLY a valid JSON object. Do NOT include any markdown fences, explanations, or extra text. The JSON must have exactly two keys:
- "label": your short category label
- "solution": your step‑by‑step solution

Ticket: {description} /no_think"""

    for attempt in range(max_retries):
        try:
            response = groq_client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=LLM_MODEL,
                temperature=0.1,
                max_tokens=500,
            )
            raw = response.choices[0].message.content
            json_str = extract_json(raw)                     # <-- robust extraction
            result = json.loads(json_str)
            label = result.get("label", "").strip()
            solution = result.get("solution") or "A solution could not be generated."
            if label:
                return {"label": label, "solution": solution}
        except Exception:
            if attempt == max_retries - 1:
                return {"label": "Uncategorised / Rare Issues",
                        "solution": "Unable to generate solution – please check the ticket manually."}
    return {"label": "Uncategorised / Rare Issues",
            "solution": "Unable to generate solution – please check the ticket manually."}

In [89]:
def classify_ticket_medoid_llm(
    description: str,
    sim_threshold: float = 0.60,
    llm_solution: bool = True,
    llm_fallback: bool = True
) -> dict:
    raw_emb = embedder.encode([description], normalize_embeddings=True).astype('float32')
    urg_feat = extract_urgency_features(description).reshape(1, -1)
    combined_emb = np.hstack([raw_emb, urg_feat])

    priority = le.inverse_transform(model.predict(combined_emb))[0]

    # 2. Medoid search (label only)
    sims = cosine_similarity(raw_emb, medoid_vectors).flatten()
    best_idx = np.argmax(sims)
    best_sim = sims[best_idx]

    label = None
    department = None
    solution = None
    source = None

    if best_sim >= sim_threshold:
        meta_id = medoid_ids[best_idx]
        label = meta_label_map[meta_id]
        department = cluster_to_department.get(meta_id, "Uncategorised")
        source = "medoid"
        if llm_solution:
            sims_all = cosine_similarity(raw_emb, inference_embs).flatten()
            top_k_idx = np.argsort(sims_all)[-5:][::-1]
            top_k_sims = sims_all[top_k_idx]
            similar_ctx = [{
                'similarity': round(float(top_k_sims[i]), 4),
                'description': inference_descs[idx][:250]
            } for i, idx in enumerate(top_k_idx)]
            solution = llm_generate_solution(description, label, priority, similar_ctx)
    else:
        if llm_fallback:
            try:
                llm_res = llm_fallback_free(description)   # no label map – free form
                label = llm_res["label"]
                solution = llm_res["solution"]
                department = "Uncategorised"
                source = "llm_fallback"
            except Exception:
                label = "Uncategorised / Rare Issues"
                department = "Uncategorised"
                solution = "LLM fallback error."
                source = "error"
        else:
            label = "Uncategorised / Rare Issues"
            department = "Uncategorised"
            solution = "No fallback configured."
            source = "none"

    if label is None or label == "":
        label = "Uncategorised / Rare Issues"
    if solution is None or solution == "":
        solution = "No solution available."
    if department is None:
        department = "Uncategorised"

    sims_all = cosine_similarity(raw_emb, inference_embs).flatten()
    top_k_idx = np.argsort(sims_all)[-5:][::-1]
    top_k_sims = sims_all[top_k_idx]
    similar = [{
        'similarity': round(float(top_k_sims[i]), 4),
        'description': inference_descs[idx][:250]
    } for i, idx in enumerate(top_k_idx)]

    return {
        "label": label,
        "department": department,
        "confidence": round(best_sim, 4),
        "priority": priority,
        "source": source,
        "solution": solution,
        "similar_tickets": similar
    }

In [90]:
ticket = 'I can''t find the Zoom recording of last week''s all‑hands meeting. Where is it stored?'

result = classify_ticket_medoid_llm(ticket, sim_threshold=0.60)
print(f"Label      : {result['label']}")
print(f"Department : {result['department']}")
print(f"Priority   : {result['priority']}")
print(f"Confidence : {result['confidence']}")
print(f"Source     : {result['source']}")
print(f"Solution   : {result['solution']}")
print(f"Similar_Tickets : {result['similar_tickets']}")

Label      : Uncategorised / Rare Issues
Department : Uncategorised
Priority   : low
Confidence : 0.49729999899864197
Source     : llm_fallback
Solution   : Unable to generate solution – please check the ticket manually.
Similar_Tickets : [{'similarity': 0.5752, 'description': 'The system is not recording all required user activities properly.'}, {'similarity': 0.5469, 'description': 'The system is not logging all required actions for compliance purposes.'}, {'similarity': 0.5465, 'description': "I need to view the logs for the 'reconciliation-batch' job that ran at 2 AM last night. The job pod has already been cleaned up. Do we have these logs stored in S3 somewhere? I need to see why the batch failed for 15 specific accounts."}, {'similarity': 0.5441, 'description': "The 'kubectl logs' command for a pod that terminated yesterday returns 'Error from server (NotFound): pods 'my-pod-xyz' not found'. The pod was deleted by the HPA. We have centralized logging, but the logs for that pod a

In [75]:
joblib.dump(embeddings_np, 'embeddings_np.pkl')          
joblib.dump(emb_raw, 'emb_raw.pkl')                      
joblib.dump(inference_embs, 'inference_embs.pkl')     
joblib.dump(inference_descs, 'inference_descs.pkl') 

['inference_descs.pkl']

In [76]:
np.save("embeddings.npy", embeddings_np)

In [78]:
joblib.dump(medoid_vectors, 'medoid_vectors.pkl')    
joblib.dump(medoid_ids, 'medoid_ids.pkl')  

['medoid_ids.pkl']

In [79]:
joblib.dump(intent_vectorizer, 'intent_vectorizer.pkl')


['intent_vectorizer.pkl']

In [80]:
joblib.dump(model, 'priority_xgb_model.pkl')           
joblib.dump(le, 'priority_label_encoder.pkl')      
joblib.dump(extract_urgency_features, 'extract_urgency_features.pkl')

['extract_urgency_features.pkl']

In [82]:
joblib.dump(umap_model, 'umap_model.pkl')


['umap_model.pkl']

In [84]:
joblib.dump(clusterer, 'clusterer.pkl')

['clusterer.pkl']

In [87]:
import json

In [88]:
with open('meta_label_map.json', 'w') as f:
    json.dump(meta_label_map, f, indent=2)

with open('cluster_to_department.json', 'w') as f:
    json.dump(cluster_to_department, f, indent=2)

In [90]:
df['cluster_id'] = labels_final
df['meta_cluster_id'] = meta_final
df['meta_label_name'] = df['meta_cluster_id'].map(meta_label_map)
df['department'] = df['meta_cluster_id'].map(cluster_to_department)

noise_mask = df['meta_cluster_id'] == -1
df.loc[noise_mask, 'meta_label_name'] = 'Uncategorised / Rare Issues'
df.loc[noise_mask, 'department'] = 'Uncategorised'

df.to_parquet('tickets_with_labels.parquet', index=False)

In [93]:
df.to_csv('final.csv', index = False)